<header>
   <p  style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'>
       Bank ClickStream - Outcome Prediction Model with Naive Bayes/Weight of Evidence
  <br>
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>

<p style="font-size:20px;font-family:Arial"><b>Introduction</b></p>

<p style="font-size:16px;font-family:Arial"> In this notebook we will predict banking product applications from clickstream event sequences using in-database Naive Bayes and
Weight of Evidence scoring on Teradata. The overall processing pipeline follows the sequence as below:</p> 
<img src="./images/naive-bayes.png" alt="naive-bayes" style="width:100%; border: 4px solid #404040; border-radius: 10px;" />
<br>


<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>1. Connect to TeradataCloud</b></p>
<p style = 'font-size:16px;font-family:Arial'>Connect to TeradataCloud using <code>create_context</code> from the teradataml Python library. </p>

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from teradataml import *
import getpass
import time

import pandas as pd
import plotly.graph_objects as go
from plotly.offline import init_notebook_mode
from collections import defaultdict
from typing import List, Dict, Tuple, Any
import colorsys
import numpy as np

from EventSequenceHelper import generate_color_palette, hex_to_rgba, generate_sankey_data, create_sankey_diagram

In [ ]:
print("Checking if this environment is ready to connect to TeradataCloud Lake...")

if os.path.exists("/home/jovyan/JupyterLabRoot/TeradataCloud/.config/.env"):
    print("Your environment parameter file exist.  Please proceed with this use case.")
    # Load all the variables from the .env file into a dictionary
    env_vars = dotenv_values("/home/jovyan/JupyterLabRoot/TeradataCloud/.config/.env")
    # Create the Context
    eng = create_context(host=env_vars.get("host"), username=env_vars.get("username"), password=env_vars.get("my_variable"))
    execute_sql('''SET query_band='DEMO=1._Bank_ClickStream_-_Outcome_Prediction_Model_with_Naive_Bayes.ipynb;' UPDATE FOR SESSION;''')
    print("Connected to TeradataCloud with:", eng)
else:
    print("Your environment has not been prepared for connecting to TeradataCloud.")
    print("Please contact the support team.")

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>2. Check Training Table</b></p>

In [ ]:
training_table = "DEMO_Bank.Session_Events_Train"

In [ ]:
df = DataFrame(in_schema("DEMO_Bank","Session_Events_Train"))

In [ ]:
df.shape

In [ ]:
#df.head(10)
df[df['Event'] == 'ApplyMortgage'].head(10)

In [ ]:
##
## CHOOSE TARGET FOR CLASSIFICATION MODEL
##
#classification_target = 'ApplyCreditCard'
#classification_target = 'ApplyAutoLoan'
#classification_target = 'ApplyCheckingAccount'
classification_target = 'ApplyMortgage'
# classification_target = 'ApplyPersonalLoan'
#classification_target = 'ApplySavingsAccount'
#classification_target = 'ApplyTeenChecking'

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>3. Model Training </b></p>
<p style = 'font-size:16px;font-family:Arial'>Build Independent Event Probabilities on outcome and non-outcome sessions<br> 
<b style = 'font-size:16px;font-family:Arial'> Session Labeling</b><br>
Each session is classified based on the presence of the target event.</p>

In [ ]:
execute_sql("""
    create volatile table outcome_sessions
    as
    (
    	select distinct UserId, SessionId
    	from {0}
    	where Event like '{1}'
    ) with data
      primary index(UserId,SessionId)
      on commit preserve rows
""".format(training_table, classification_target)
)

<p style = 'font-size:16px;font-family:Arial'>Sessions that contain the target event.
All other events become positive training signals.</p>

In [ ]:
df = DataFrame("outcome_sessions")
df

In [ ]:
execute_sql("""
    create volatile table non_outcome_sessions
    as
    (
       select distinct UserId, SessionId
       from {0}
       where (UserId, SessionId) not in (select * from outcome_sessions)
    ) with data
    primary index(UserId,SessionId)
    on commit preserve rows
""".format(training_table)
)

<p style = 'font-size:16px;font-family:Arial'>Sessions that never lead to the target.
Used as the negative class baseline.</p>

<hr style="height:1px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>3.1 Model Tables Creation </b></p>
<p style = 'font-size:16px;font-family:Arial'>Create a row for every (event, class) pair: 150 events × 2 classes = 300 rows. Initialize all counts to zero. 

In [ ]:
execute_sql("""
    create volatile table all_unique_events as
    (
    	select distinct Event as Event from {0}
    )
    with data
    primary index(Event)
    on commit preserve rows
""".format(training_table)
)

In [ ]:
df = DataFrame("all_unique_events")
df

In [ ]:
execute_sql("""
    create volatile table base_model as
    (
    	select cast(1 as integer) as outcome, Event as a, cast(0 as float) as counts, cast(0 as float) as probability
    	from all_unique_events
    	union all
    	select cast(0 as integer) as outcome, Event as a, cast(0 as float) as counts, cast(0 as float) as probability
    	from all_unique_events
    )
    with data
    primary index(a, outcome)
    on commit preserve rows
""")

<hr style="height:1px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>3.2 Update Model Table with counts and normalize with laplacians </b></p>
<p style = 'font-size:16px;font-family:Arial'>For each event, count distinct sessions where it appears — separately for outcome and non-outcome. Exclude the target event. </p>

In [ ]:
execute_sql("""
    update base_model
    from (
        select Event, 
               count(distinct UserId || '-' || cast(SessionId as varchar(10))) as counts
        from {0}
        where (UserId, SessionId) in (select UserId, SessionId from outcome_sessions)
          and Event not like '{1}'
        group by 1
    ) x
    set counts = x.counts
    where base_model.outcome = 1
      and base_model.a = x.Event
""".format(training_table,classification_target)
)

execute_sql("""
    update base_model
    from (
    	 select Event, count (distinct (UserId || '-' || CAST(SessionId AS VARCHAR(10)))) as counts
    	 from {0} a
    	 where (UserId, SessionId) in (select UserId, SessionId from non_outcome_sessions) 
    	 group by 1
    ) x
    set counts = x.counts
    where base_model.outcome = 0
      and base_model.a = x.Event
""".format(training_table)
)

In [ ]:
df = DataFrame("base_model")
df

<p style = 'font-size:16px;font-family:Arial'>Update Base Model outcome/no-outcome probabilities from counts with laplacian to account for zeroes. Add +1 to every count. Prevents zero probabilities for events never seen in one class.</p>

In [ ]:
## Add laplacian 
execute_sql("""
    update base_model set counts = counts + 1
""")

## Fix the probability column for both outcome and non-outcome across all events and outcome/no-outcome
execute_sql("""
    update base_model 
    from ( select sum(counts) as total from base_model where outcome = 1) as x
    set probability = counts/x.total
    where outcome = 1
;
""")

execute_sql("""
    update base_model 
    from ( select sum(counts) as total from base_model where outcome = 0) as x
    set probability = counts/x.total
    where outcome = 0
;
""")

In [ ]:
df = DataFrame("base_model").to_pandas().reset_index()

In [ ]:
df[df['outcome'] == 1].sort_values(by='probability',ascending=False)

In [ ]:
df[df['outcome'] == 0].sort_values(by='probability',ascending=False)

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>4. Prepare Test Set with Ground Truth Separation and run Prediction Logic</b></p>

In [ ]:
test_holdout_table = "DEMO_Bank.Session_Events_Test"

In [ ]:
df = DataFrame(in_schema("DEMO_Bank","Session_Events_Test"))

In [ ]:
df.shape

In [ ]:
df.head(10)

In [ ]:
execute_sql("""
    create volatile table test_outcome_sessions
    as
    (
    	select distinct UserId, SessionId
    	from {0}
    	where Event like '{1}'
    ) with data
    primary index(UserId,SessionId)
    on commit preserve rows
""".format(test_holdout_table, classification_target)
)

In [ ]:
execute_sql("""
    create volatile table test_non_outcome_sessions
    as
    (
       select distinct UserId, SessionId
       from {0}
       where (UserId, SessionId) not in (select * from test_outcome_sessions)
    ) with data
    primary index(UserId,SessionId)
    on commit preserve rows
""".format(test_holdout_table)
)

<p style = 'font-size:16px;font-family:Arial'><b>Scoring: The Math</b><br>
    <b>Log-Odds (per session)</b><br>
<code>log_odds = Σ log( P(event_i | outcome) / P(event_i | non-outcome) ) </code><br>
Sum over all events in the session. Each event contributes its Weight of Evidence.<br>
<b>Sigmoid Conversion</b><br>
<code>score = 1 / (1 + exp(-log_odds))</code>
Maps raw log-odds into a 0–1 probability for classification.<br>
Key Insight: Each event is treated independently (the “Naive” assumption). Mortgage-related events contribute positive WoE;
routine banking events contribute negative WoE. The session score is the cumulative evidence.

In [ ]:
execute_sql("""
     create volatile table test_scored_table
     as
     (
       select UserId, SessionId, events_per_session, log_odds, 1 / (1 + EXP(-log_odds)) AS score -- doing a sigmoid here
       from(
            select UserId, SessionId, count(*) as events_per_session, sum(log(o.probability/n.probability)) as log_odds
            from {0} a,
                 base_model o,
                 base_model n
            where o.outcome = 1 and 
                  n.outcome = 0 and
                  a.Event = o.a and
                  a.Event = n.a
            group by 1,2
        ) x
     ) with data
     primary index(UserId,SessionId)
     on commit preserve rows
""".format(test_holdout_table)
)

<p style="font-size:16px;font-family:Arial">The entire scoring computation runs as a single SQL query inside Teradata. The base_model table acts as a lightweight lookup — no Python-side
prediction loop needed.</p>

In [ ]:
df = DataFrame("test_scored_table")

In [ ]:
df[df['log_odds'] > 0]

In [ ]:
df[df['log_odds'] < 0]

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>5. Evaluation wrt Ground Truth/Precision Recall Metrics</b></p>

In [ ]:
execute_sql("""
    create volatile table test_evaluation_table
    as
    (
        select a.UserId, a.SessionId, a.events_per_session, a.score, b.outcome
        from
            test_scored_table a
            inner join
            (
              select 1 as outcome, UserId, SessionId 
                   from test_outcome_sessions
              union all
              select 0 as outcome, UserId, SessionId 
                   from test_non_outcome_sessions
            ) b on (a.UserId = b.UserId and a.SessionId = b.SessionId)
    ) with data
    primary index(UserId,SessionId)
    on commit preserve rows
""")

In [ ]:
df = DataFrame("test_evaluation_table").to_pandas().reset_index()

In [ ]:
df.shape

In [ ]:
df

In [ ]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
dfp = df[df["events_per_session"] > 4]

In [ ]:
thresholds = [0.1, 0.5, 0.7, 0.999]
pythonfig, axes = plt.subplots(1, len(thresholds), figsize=(4*len(thresholds), 4))

for ax, threshold in zip(axes, thresholds):
    predicted = (dfp['score'] >= threshold).astype(int)
    cm = confusion_matrix(dfp['outcome'], predicted)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title(f'Threshold = {threshold}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
def confusion_matrix_manual(y_true, y_pred):
    tp = ((y_true == 1) & (y_pred == 1)).sum()
    tn = ((y_true == 0) & (y_pred == 0)).sum()
    fp = ((y_true == 0) & (y_pred == 1)).sum()
    fn = ((y_true == 1) & (y_pred == 0)).sum()
    return np.array([[tn, fp], [fn, tp]])



for threshold in thresholds:
    # Convert probabilities to binary predictions
    predicted = (df['score'] >= threshold).astype(int)
    
    # Generate confusion matrix
    cm = confusion_matrix_manual(df['outcome'], predicted)
    
    tn, fp, fn, tp = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    
    print(f"\n--- Threshold: {threshold} ---")
    print(f"Confusion Matrix:")
    print(f"              Predicted 0  Predicted 1")
    print(f"Actual 0      {tn:>10}  {fp:>10}")
    print(f"Actual 1      {fn:>10}  {tp:>10}")
    print(f"Accuracy:  {accuracy:.3f}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall:    {recall:.3f}")

In [ ]:
df.groupby('outcome')['score'].describe()

In [ ]:
df.groupby('outcome')['score'].describe()

In [ ]:
df[df['outcome'] == 1]['score'].mean()  # Should be higher

In [ ]:
df[df['outcome'] == 0]['score'].mean()  # Should be lower

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>6. Multi-Class Classification</b></p>
<p style="font-size:16px;font-family:Arial">
Instead of asking <b>Will they apply for a mortgage?</b> (binary), we now ask <b>"Which product will they apply for?"</b> (multiclass).<br>
    <b>Approach:</b> Train one binary model per Apply/ target class. For each test session, score it against all class models.<br>
The predicted class is the one with the highest log-likelihood ratio (log-odds). Sessions with all negative log-odds are classified as NoApplication.
    <br><b>Target classes discovered automatically</b> from the training data.

In [ ]:
# Discover all Apply* events from training data
apply_events_df = DataFrame.from_query("""
    SELECT DISTINCT Event FROM {0} WHERE Event LIKE 'Apply%'
""".format(training_table))
apply_targets = apply_events_df.to_pandas()['Event'].tolist()
print(f"Discovered {len(apply_targets)} target classes: {apply_targets}")

In [ ]:
# Train one Naive Bayes model per target class
# Each model is stored as a permanent table: nb_model_<ClassName>

for target in apply_targets:
    model_table = f"nb_model_{target}"
    print(f"\nTraining Naive Bayes for: {target}")
    
    # Drop if exists
    try:
        db_drop_table(model_table)
    except:
        pass
    
    # Outcome sessions for this target
    execute_sql(f"""
        CREATE VOLATILE TABLE mc_outcome_sess AS (
            SELECT DISTINCT UserId, SessionId FROM {training_table}
            WHERE Event = '{target}'
        ) WITH DATA PRIMARY INDEX(UserId, SessionId) ON COMMIT PRESERVE ROWS
    """)
    
    execute_sql(f"""
        CREATE VOLATILE TABLE mc_non_outcome_sess AS (
            SELECT DISTINCT UserId, SessionId FROM {training_table}
            WHERE (UserId, SessionId) NOT IN (SELECT * FROM mc_outcome_sess)
        ) WITH DATA PRIMARY INDEX(UserId, SessionId) ON COMMIT PRESERVE ROWS
    """)
    
    # Build base model: event x outcome(0/1)
    execute_sql(f"""
        CREATE TABLE {model_table} , STORAGE = TD_OFSSTORAGE AS (
            SELECT CAST(1 AS INTEGER) AS outcome, Event AS a,
                   CAST(0 AS FLOAT) AS counts, CAST(0 AS FLOAT) AS probability
            FROM all_unique_events
            UNION ALL
            SELECT CAST(0 AS INTEGER) AS outcome, Event AS a,
                   CAST(0 AS FLOAT) AS counts, CAST(0 AS FLOAT) AS probability
            FROM all_unique_events
        ) WITH DATA PRIMARY INDEX(a, outcome)
    """)
    
    # Count events in outcome sessions
    execute_sql(f"""
        UPDATE {model_table}
        FROM (
            SELECT Event, COUNT(DISTINCT UserId || '-' || CAST(SessionId AS VARCHAR(10))) AS counts
            FROM {training_table}
            WHERE (UserId, SessionId) IN (SELECT UserId, SessionId FROM mc_outcome_sess)
              AND Event NOT LIKE 'Apply%'
            GROUP BY 1
        ) x
        SET counts = x.counts
        WHERE {model_table}.outcome = 1 AND {model_table}.a = x.Event
    """)
    
    # Count events in non-outcome sessions
    execute_sql(f"""
        UPDATE {model_table}
        FROM (
            SELECT Event, COUNT(DISTINCT UserId || '-' || CAST(SessionId AS VARCHAR(10))) AS counts
            FROM {training_table}
            WHERE (UserId, SessionId) IN (SELECT UserId, SessionId FROM mc_non_outcome_sess)
            GROUP BY 1
        ) x
        SET counts = x.counts
        WHERE {model_table}.outcome = 0 AND {model_table}.a = x.Event
    """)
    
    # Laplacian smoothing + normalize
    execute_sql(f"UPDATE {model_table} SET counts = counts + 1")
    execute_sql(f"""
        UPDATE {model_table}
        FROM (SELECT SUM(counts) AS total FROM {model_table} WHERE outcome = 1) AS x
        SET probability = counts / x.total WHERE outcome = 1
    """)
    execute_sql(f"""
        UPDATE {model_table}
        FROM (SELECT SUM(counts) AS total FROM {model_table} WHERE outcome = 0) AS x
        SET probability = counts / x.total WHERE outcome = 0
    """)
    
    # Cleanup volatile tables
    try:
        execute_sql("DROP TABLE mc_outcome_sess")
        execute_sql("DROP TABLE mc_non_outcome_sess")
    except:
        pass
    
    cnt = DataFrame(model_table).shape[0]
    print(f"  {model_table}: {cnt} rows")

print(f"\nDone. Trained {len(apply_targets)} Naive Bayes models.")

In [ ]:
# Score all test sessions against each class model
# For each class: compute log-odds = SUM(log(P(event|outcome) / P(event|non-outcome)))

score_parts = []
for target in apply_targets:
    model_table = f"nb_model_{target}"
    score_parts.append(f"""
        SELECT UserId, SessionId, CAST('{target}' AS VARCHAR(50)) AS target_class,
               SUM(LOG(o.probability / n.probability)) AS log_odds,
               1.0 / (1.0 + EXP(-SUM(LOG(o.probability / n.probability)))) AS score
        FROM {test_holdout_table} a
        JOIN {model_table} o ON o.outcome = 1 AND a.Event = o.a
        JOIN {model_table} n ON n.outcome = 0 AND a.Event = n.a
        GROUP BY UserId, SessionId
    """)

union_sql = " UNION ALL ".join(score_parts)

try:
    execute_sql("DROP TABLE mc_nb_all_scores")
except:
    pass

execute_sql(f"""
    CREATE TABLE mc_nb_all_scores,STORAGE = TD_OFSSTORAGE  AS (
        {union_sql}
    ) WITH DATA PRIMARY INDEX(UserId, SessionId, target_class)
""")

print(f"Scored {DataFrame('mc_nb_all_scores').shape[0]} (session, class) pairs")

In [ ]:
# Classify: each session gets the class with the highest log-odds
# Sessions where best log-odds < 0 => NoApplication

try:
    execute_sql("DROP TABLE mc_predictions")
except:
    pass

execute_sql("""
    CREATE TABLE mc_predictions,STORAGE = TD_OFSSTORAGE  AS (
        SELECT UserId, SessionId, target_class AS predicted_class, log_odds, score
        FROM mc_nb_all_scores
        QUALIFY ROW_NUMBER() OVER (PARTITION BY UserId, SessionId ORDER BY log_odds DESC) = 1
    ) WITH DATA PRIMARY INDEX(UserId, SessionId)
""")

# Override: if best log_odds < 0, classify as NoApplication
execute_sql("""
    UPDATE mc_predictions SET predicted_class = 'NoApplication' WHERE log_odds < 0
""")

print("Predictions:")
pred_df = DataFrame('mc_predictions').to_pandas()
print(pred_df['predicted_class'].value_counts())

In [ ]:
# Build ground truth: actual class per test session
try:
    execute_sql("DROP TABLE mc_ground_truth")
except:
    pass

execute_sql(f"""
    CREATE TABLE mc_ground_truth,STORAGE = TD_OFSSTORAGE  AS (
        SELECT UserId, SessionId,
               CAST(COALESCE(MAX(CASE WHEN Event LIKE 'Apply%' THEN Event END), 'NoApplication') AS VARCHAR(50)) AS true_class
        FROM {test_holdout_table}
        GROUP BY UserId, SessionId
    ) WITH DATA PRIMARY INDEX(UserId, SessionId)
""")

gt_df = DataFrame('mc_ground_truth').to_pandas()
print("Ground truth distribution:")
print(gt_df['true_class'].value_counts())

In [ ]:
# Join predictions with ground truth and evaluate
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

eval_df = DataFrame.from_query("""
    SELECT g.UserId, g.SessionId, g.true_class, 
           COALESCE(p.predicted_class, 'NoApplication') AS predicted_class
    FROM mc_ground_truth g
    LEFT JOIN mc_predictions p ON g.UserId = p.UserId AND g.SessionId = p.SessionId
""").to_pandas()

print(f"\nTotal test sessions evaluated: {len(eval_df)}")
print(f"\nOverall Accuracy: {accuracy_score(eval_df['true_class'], eval_df['predicted_class']):.4f}")
print(f"\nClassification Report:")
print(classification_report(eval_df['true_class'], eval_df['predicted_class'], zero_division=0))

In [ ]:
# Multiclass confusion matrix heatmap
import matplotlib.pyplot as plt
import seaborn as sns

labels = sorted(eval_df['true_class'].unique())
cm = confusion_matrix(eval_df['true_class'], eval_df['predicted_class'], labels=labels)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=axes[0])
axes[0].set_title('Confusion Matrix (Counts)')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
plt.setp(axes[0].get_xticklabels(), rotation=45, ha='right', fontsize=8)

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=axes[1])
axes[1].set_title('Confusion Matrix (Normalized)')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
plt.setp(axes[1].get_xticklabels(), rotation=45, ha='right', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Per-class precision, recall, F1 summary table
from sklearn.metrics import precision_recall_fscore_support

p, r, f1, sup = precision_recall_fscore_support(
    eval_df['true_class'], eval_df['predicted_class'], labels=labels, zero_division=0
)
summary = pd.DataFrame({
    'Class': labels, 'Precision': p, 'Recall': r, 'F1': f1, 'Support': sup
}).set_index('Class')
print(summary.to_string())
print(f"\nWeighted F1: {(summary['F1'] * summary['Support']).sum() / summary['Support'].sum():.4f}")

<hr style="height:2px;border:none;">
<p style = 'font-size:20px;font-family:Arial'><b>4. Cleanup </b></p>

In [ ]:
# Cleanup multiclass tables (optional)
for target in apply_targets:
    try: db_drop_table(f"nb_model_{target}")
    except: pass
for tbl in ['mc_nb_all_scores', 'mc_predictions', 'mc_ground_truth']:
    try: db_drop_table(tbl)
    except: pass

In [ ]:
remove_context()

<footer style="padding-bottom:35px; border-bottom:3px solid">
  <div style="float:right; margin-top:14px">Copyright © Teradata - 2026. All Rights Reserved</div>
</footer>